In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed# Install CatBoost library
!pip install catboost -q

import pandas as pd
import numpy as np
import os
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping
from catboost import CatBoostClassifier

# Configuration
warnings.filterwarnings('ignore')
print("✅ Libraries Imported.")

/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():


✅ Libraries Imported.


In [2]:
# Initialize paths
train_path, test_path, sub_path = '', '', ''

# Search for dataset in Kaggle input directory
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        full_path = os.path.join(dirname, filename)
        if 'train.csv' in filename: train_path = full_path
        elif 'test.csv' in filename: test_path = full_path
        elif 'sample_submission.csv' in filename: sub_path = full_path

# Load Data
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
submission = pd.read_csv(sub_path)

print(f"Data Loaded. Train: {train.shape}, Test: {test.shape}")

Data Loaded. Train: (700000, 26), Test: (300000, 25)


In [3]:
# 1. Separate ID and Target
test_ids = test['id']
train = train.drop('id', axis=1)
test = test.drop('id', axis=1)

target = train['diagnosed_diabetes']
train = train.drop('diagnosed_diabetes', axis=1)

# 2. One-Hot Encoding
train_len = len(train)
combined = pd.concat([train, test], axis=0)
combined_encoded = pd.get_dummies(combined, drop_first=True)

# 3. Split back to Train and Test
X = combined_encoded.iloc[:train_len]
X_test = combined_encoded.iloc[train_len:]

# 4. Scaling (Standardization)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for easier handling
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print("Preprocessing Complete.")

Preprocessing Complete.


In [4]:
# K-Fold Configuration
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# Arrays to store predictions
xgb_oof = np.zeros(len(X))
lgbm_oof = np.zeros(len(X))
cat_oof = np.zeros(len(X))

xgb_test_pred = np.zeros(len(X_test))
lgbm_test_pred = np.zeros(len(X_test))
cat_test_pred = np.zeros(len(X_test))

print(f"Starting {N_SPLITS}-Fold Training...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_scaled, target)):
    print(f"\n[Fold {fold + 1}/{N_SPLITS}]")
    
    # Split Data for this fold
    X_tr, X_val = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
    y_tr, y_val = target.iloc[train_idx], target.iloc[val_idx]
    
    # --- 1. XGBoost ---
    xgb = XGBClassifier(
        n_estimators=2000, learning_rate=0.015, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, eval_metric='auc',
        random_state=42, n_jobs=-1, early_stopping_rounds=100
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=0)
    
    xgb_oof[val_idx] = xgb.predict_proba(X_val)[:, 1]
    xgb_test_pred += xgb.predict_proba(X_test_scaled)[:, 1] / N_SPLITS
    print(f"  XGB AUC: {roc_auc_score(y_val, xgb_oof[val_idx]):.5f}")

    # --- 2. LightGBM ---
    lgbm = LGBMClassifier(
        n_estimators=2000, learning_rate=0.015, num_leaves=31,
        subsample=0.8, colsample_bytree=0.8, metric='auc',
        random_state=42, n_jobs=-1, verbosity=-1
    )
    lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], 
             callbacks=[early_stopping(100, verbose=False)])
    
    lgbm_oof[val_idx] = lgbm.predict_proba(X_val)[:, 1]
    lgbm_test_pred += lgbm.predict_proba(X_test_scaled)[:, 1] / N_SPLITS
    print(f"  ⚡ LGB AUC: {roc_auc_score(y_val, lgbm_oof[val_idx]):.5f}")

    # --- 3. CatBoost ---
    cat = CatBoostClassifier(
        iterations=2000, learning_rate=0.015, depth=6,
        eval_metric='AUC', random_seed=42, verbose=0,
        early_stopping_rounds=100, allow_writing_files=False
    )
    cat.fit(X_tr, y_tr, eval_set=(X_val, y_val))
    
    cat_oof[val_idx] = cat.predict_proba(X_val)[:, 1]
    cat_test_pred += cat.predict_proba(X_test_scaled)[:, 1] / N_SPLITS
    print(f"  Cat AUC: {roc_auc_score(y_val, cat_oof[val_idx]):.5f}")

print("\nTraining Complete!")

Starting 5-Fold Training...

[Fold 1/5]
  XGB AUC: 0.72646
  ⚡ LGB AUC: 0.72694
  Cat AUC: 0.72291

[Fold 2/5]
  XGB AUC: 0.72442
  ⚡ LGB AUC: 0.72461
  Cat AUC: 0.72090

[Fold 3/5]
  XGB AUC: 0.72527
  ⚡ LGB AUC: 0.72571
  Cat AUC: 0.72183

[Fold 4/5]
  XGB AUC: 0.72689
  ⚡ LGB AUC: 0.72664
  Cat AUC: 0.72282

[Fold 5/5]
  XGB AUC: 0.72598
  ⚡ LGB AUC: 0.72655
  Cat AUC: 0.72244

Training Complete!


In [5]:
from scipy.optimize import minimize

oof_preds = pd.DataFrame({
    'xgb': xgb_oof,
    'lgbm': lgbm_oof,
    'cat': cat_oof
})

test_preds = pd.DataFrame({
    'xgb': xgb_test_pred,
    'lgbm': lgbm_test_pred,
    'cat': cat_test_pred
})


def minimize_auc(weights):
  
    weights = np.array(weights)
    weights /= weights.sum()
   
    final_oof = (oof_preds['xgb'] * weights[0]) + \
                (oof_preds['lgbm'] * weights[1]) + \
                (oof_preds['cat'] * weights[2])
    
   
    return -roc_auc_score(target, final_oof)


initial_weights = [0.33, 0.33, 0.33]
result = minimize(minimize_auc, initial_weights, method='Nelder-Mead')


best_weights = result.x / result.x.sum()
print(f"  XGBoost : {best_weights[0]:.5f}")
print(f"  LightGBM: {best_weights[1]:.5f}")
print(f"  CatBoost: {best_weights[2]:.5f}")


final_oof_score = -result.fun
print(f"CV Score: {final_oof_score:.5f}")


final_pred = (test_preds['xgb'] * best_weights[0]) + \
             (test_preds['lgbm'] * best_weights[1]) + \
             (test_preds['cat'] * best_weights[2])


submission_df = pd.DataFrame({
    'id': test_ids,
    'diagnosed_diabetes': final_pred
})

submission_df.to_csv('submission_optimized.csv', index=False)
print("'submission_optimized.csv' 생성 완료!")
display(submission_df.head())

  XGBoost : 0.58013
  LightGBM: 0.96057
  CatBoost: -0.54070
CV Score: 0.72686
'submission_optimized.csv' 생성 완료!


,id,diagnosed_diabetes
0,700000,0.495244
1,700001,0.688500
2,700002,0.768514
3,700003,0.378251
4,700004,0.918830
